# T53-yy区域数据整理

## 任务概述

本工具用于从企业预警通(Ratingdog)平台获取区域经济数据，主要功能包括：

1. **区域数据采集**：从企业预警通API获取各级行政区域的经济数据
2. **多维度指标**：涵盖GDP、财政收支、债务限额、土地成交等50+指标
3. **行政区域归整**：将复杂的区域名称标准化为省-市-区县三级结构
4. **数据自动更新**：支持增量更新，自动添加新字段

---

## 1. 环境配置与依赖导入

### 1.1 导入必要的库

In [ ]:
# 标准库
import json
import os
import sys
import re
from time import sleep
from datetime import datetime, timedelta

# 第三方库
import requests
import pandas as pd
import sqlalchemy
from sqlalchemy import text

# 配置导入
from config import (
    RATINGDOG_USERNAME,
    RATINGDOG_PASSWORD,
    RATINGDOG_PHONE_CODE,
    DB_HOST,
    DB_PORT,
    DB_USER,
    DB_PASSWORD,
    DB_NAME,
    get_database_engine,
    get_api_headers
)

print("环境配置完成")

### 1.2 配置说明

敏感信息已通过环境变量配置，请确保以下环境变量已设置：
- `RATINGDOG_USERNAME`: 企业预警通用户名
- `RATINGDOG_PASSWORD`: 企业预警通密码
- `DB_HOST`: 数据库主机地址
- `DB_USER`: 数据库用户名
- `DB_PASSWORD`: 数据库密码

In [ ]:
# 初始化数据库连接
sql_engine = get_database_engine()
print(f"数据库连接已创建: {DB_HOST}:{DB_PORT}/{DB_NAME}")

---

## 2. 数据源分析

### 2.1 企业预警通API认证

企业预警通使用Bearer Token认证方式。

In [ ]:
def login_ratingdog():
    """
    登录企业预警通获取AccessToken
    
    Returns:
        str: AccessToken用于后续API调用
    """
    url_login = 'https://auth.ratingdog.cn/api/TokenAuth/Authenticate'
    
    headers_login = {
        '.Aspnetcore.Culture': 'zh-Hans',
        'Accept': 'application/json, text/plain, */*',
        'Content-Type': 'application/json;charset=UTF-8',
        'Devicechannel': 'gclife_bmp_pc',
        'Origin': 'https://www.ratingdog.cn',
        'Ratingdog.Tenantid': '114',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    data_login = {
        "UserNameOrEmailAddressOrPhone": RATINGDOG_USERNAME,
        "internationalPhoneCode": RATINGDOG_PHONE_CODE,
        "password": RATINGDOG_PASSWORD
    }
    
    response = requests.post(url_login, headers=headers_login, json=data_login)
    result = response.json()
    
    if 'result' in result and 'accessToken' in result['result']:
        access_token = result['result']['accessToken']
        print(f"登录成功，Token获取完成")
        return access_token
    else:
        raise Exception(f"登录失败: {result}")

# 获取AccessToken
access_token = login_ratingdog()

### 2.2 API请求头配置

In [ ]:
# 获取完整的API请求头
headers_yy = get_api_headers(access_token)
print("API请求头配置完成")

### 2.3 数据指标说明

企业预警通提供多维度区域经济数据，包括：

| 数据类别 | DataCategoryType | 主要指标 |
|---------|-----------------|---------|
| 经济数据 | 1001 | GDP、固定资产投资、常住人口等 |
| 财政数据 | 1002 | 一般预算收入、税收收入、债务限额等 |
| 金融数据 | 1003 | 存款余额、贷款余额、不良贷款率等 |

---

## 3. 数据获取与处理

### 3.1 定义指标代码列表

In [ ]:
# 经济指标
ECONOMIC_INDICATORS = [
    "GDP", "GDPGrowthRate", "GDPPerCaptita", "InvestmentInFixedAssets",
    "InvestmentInFixedAssetsGrowthRate", "PermanentResidents", "GDPPerCaptita_model"
]

# 财政指标
FISCAL_INDICATORS = [
    "FiscalSelfsufficiencyRate", "TaxPercentage", "GeneralBudgetRevenue",
    "GeneralBudgetRevenueGrowthRate", "TaxIncome", "TaxIncomeGrowthRate",
    "TransferIncome", "SubsidyIncomeFromSuperiors", "TotalGeneralPublicBudgetRevenue",
    "GeneralBudgetExpenditure", "GeneralBudgetExpenditureGrowthRate",
    "GeneralBudgetRevenueGrowthRate_NaturalCaliber", "TaxIncome_NaturalCaliber",
    "GovernmentFundIncome", "GovernmentFundIncomeGrowthRate", "LandSaleRevenue",
    "GovernmentFundExpenditure", "GeneralDebtLimit", "SpecialDebtLimit",
    "GeneralDebtBalance", "SpecialDebtBalance"
]

# 金融指标
FINANCIAL_INDICATORS = [
    "RMBBalanceOfDeposits", "ForeignCurrencyBalanceOfDeposits",
    "RMBBalanceOfLoad", "ForeignCurrencyBalanceOfLoad",
    "TotalFinancialAssets", "NonperformingLoanRatio"
]

# 隐性债务指标
IMPLICIT_DEBT_INDICATORS = [
    "ImplicitDebtResolution", "YYRatio", "InterestDebt", "SurvivalAmount",
    "ThreeYearsCompositeGrowthRate", "FiveYearsCompositeGrowthRate",
    "InterestDebtYY", "InterestDebtYYBankLoan", "InterestDebtYYBonds",
    "InterestDebtYYNonStandard", "InterestDebtYYOther"
]

# 房地产指标
REAL_ESTATE_INDICATORS = [
    "MHPI_YoY", "MHPI_MoM", "ResidentialHouseTradeArea",
    "ResidentialHouseTradeAmount", "ResidentialHouseTradeAvgPrice",
    "SecondhandResidentialHouseTradeArea"
]

# 城市基础设施指标
INFRASTRUCTURE_INDICATORS = [
    "UrbanArea", "UrbanPopulation", "MetropolitanArea", "MetropolitanPopulation",
    "BuiltUpArea", "InvestInFAIOfCMPF", "NewFixedAssetOfCMPF",
    "LengthOfRoad", "RoadLengthInBuiltUpArea", "RoadSurface", "RoadAreaInBuiltUpArea",
    "NumberOfBridge", "MajorBridgesAndSuperMajorBridges",
    "ComprehensiveProductionCapacityOfTapWater", "LengthOfWaterSupplyPipeline",
    "WaterPopulation", "LengthOfDrainagePipe", "TotalSewageDischarge", "TotalSewageTreatment"
]

# 合并所有指标
ALL_INDICATORS = (
    ECONOMIC_INDICATORS + FISCAL_INDICATORS + FINANCIAL_INDICATORS +
    IMPLICIT_DEBT_INDICATORS + REAL_ESTATE_INDICATORS + INFRASTRUCTURE_INDICATORS
)

print(f"共定义 {len(ALL_INDICATORS)} 个数据指标")

### 3.2 数据采集核心函数

In [ ]:
def fetch_region_data(headers, max_result_count=150, skip_count=0, 
                       data_category_types=[1001, 1002, 1003],
                       category_codes=None):
    """
    从企业预警通获取区域数据
    
    Args:
        headers: API请求头
        max_result_count: 每页最大记录数
        skip_count: 跳过的记录数（分页）
        data_category_types: 数据类别列表
        category_codes: 指标代码列表
    
    Returns:
        tuple: (DataFrame数据, 总记录数, 表头信息)
    """
    if category_codes is None:
        category_codes = ALL_INDICATORS
    
    url = "https://host.ratingdog.cn/api/services/app/AdministrativeDivisionData/GetAdministrativeDivisionDatas"
    
    data = {
        "AdministrativeDivisionIds": [],
        "ReportDates": [],
        "AdministrativeDivisionLevels": [],
        "DataCategoryTypes": data_category_types,
        "categoryCodes": category_codes,
        "MaxResultCount": max_result_count,
        "SkipCount": skip_count
    }
    
    response = requests.post(url, headers=headers, json=data)
    response_text = str(response.content, encoding="utf-8")
    result = json.loads(response_text)
    
    # 提取表头和数据
    table_header = result['result']['tableHeader']
    table_content = result['result']['tableData']['items']
    total_count = result['result']['tableData']['totalCount']
    
    # 构建DataFrame
    column_names = [col for col in table_header.keys()]
    rename_columns = {col: table_header[col]['name'] for col in table_header}
    
    df = pd.DataFrame(table_content, columns=column_names)
    df.rename(rename_columns, inplace=True, axis='columns')
    
    return df, total_count, table_header

print("数据采集函数定义完成")

### 3.3 动态字段管理

企业预警通可能随时新增指标，需要动态更新数据库表结构。

In [ ]:
def update_table_schema(engine, table_name, new_columns):
    """
    动态更新数据库表结构，添加新字段
    
    Args:
        engine: SQLAlchemy引擎
        table_name: 表名
        new_columns: 需要添加的新列列表
    
    Returns:
        bool: 是否成功添加新列
    """
    if not new_columns:
        return False
    
    try:
        # 构造ALTER TABLE SQL语句
        alter_sql = f"ALTER TABLE {table_name} ADD COLUMN {', '.join(f'{col} TEXT' for col in new_columns)}"
        with engine.begin() as connection:
            connection.execute(text(alter_sql))
        print(f"成功添加新列: {new_columns}")
        return True
    except Exception as e:
        print(f"添加列失败: {e}")
        return False

def check_and_update_schema(engine, table_name, df):
    """
    检查并更新表结构
    
    Args:
        engine: SQLAlchemy引擎
        table_name: 表名
        df: 新数据DataFrame
    """
    new_columns = df.columns.tolist()
    
    try:
        with engine.begin() as connection:
            existing_df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 1", connection)
            existing_columns = existing_df.columns.tolist()
            
        # 找出新列
        columns_to_add = [col for col in new_columns if col not in existing_columns]
        
        if columns_to_add:
            print(f"发现新字段: {columns_to_add}")
            update_table_schema(engine, table_name, columns_to_add)
    except Exception as e:
        print(f"表结构检查异常(可能是新表): {e}")

print("动态字段管理函数定义完成")

### 3.4 执行数据采集

In [ ]:
def collect_all_region_data(headers, engine, table_name='yq.yyqy'):
    """
    采集所有区域数据（分页处理）
    
    Args:
        headers: API请求头
        engine: 数据库引擎
        table_name: 目标表名
    
    Returns:
        int: 总采集记录数
    """
    skip_count = 0
    is_end = False
    total_records = 0
    page_size = 150
    
    while not is_end:
        print(f"正在获取数据，SkipCount={skip_count}...")
        
        # 获取数据
        df, total_count, _ = fetch_region_data(
            headers=headers,
            max_result_count=page_size,
            skip_count=skip_count
        )
        
        # 检查并更新表结构
        if skip_count == 0:
            check_and_update_schema(engine, table_name, df)
        
        # 保存数据
        with engine.begin() as connection:
            df.to_sql(name=table_name.split('.')[-1], con=connection, if_exists='append', index=False)
        
        total_records += len(df)
        print(f"已保存 {len(df)} 条记录，累计 {total_records} 条")
        
        # 判断是否结束
        if skip_count + page_size >= total_count:
            is_end = True
            print(f"数据采集完成，共 {total_count} 条记录")
        else:
            skip_count += page_size
            sleep(0.5)  # 避免请求过快
    
    return total_records

# 执行数据采集
total = collect_all_region_data(headers_yy, sql_engine, 'yq.yyqy')
print(f"数据采集完成，共获取 {total} 条记录")

---

## 4. 核心算法逻辑

### 4.1 行政区域标准化

企业预警通的区域名称格式为`{省}-{市}-{区县}`，需要进行标准化处理。

In [ ]:
# 直辖市列表
DIRECT_MUNICIPALITIES = ['北京市', '上海市', '天津市', '重庆市']

# 省直辖县级行政区划列表
PROVINCE_DIRECT_COUNTIES = [
    '五家渠市', '北屯市', '双河市', '可克达拉市', '图木舒克市', '新星市', '昆玉市',
    '石河子市', '胡杨河市', '铁门关市', '阿拉尔市', '济源市', '万宁市', '东方市',
    '临高县', '乐东黎族自治县', '五指山市', '保亭黎族苗族自治县', '定安县', '屯昌县',
    '文昌市', '昌江黎族自治县', '澄迈县', '琼中黎族苗族自治县', '琼海市', 
    '白沙黎族自治县', '陵水黎族自治县', '仙桃市', '天门市', '潜江市', '神农架林区'
]

def is_direct_municipality(region_name):
    """检查是否为直辖市"""
    return any(region_name.startswith(city) for city in DIRECT_MUNICIPALITIES)

def is_province_direct_county(region_name):
    """检查是否为省直辖县级行政区划"""
    return any(county in region_name for county in PROVINCE_DIRECT_COUNTIES)

print(f"定义了 {len(DIRECT_MUNICIPALITIES)} 个直辖市")
print(f"定义了 {len(PROVINCE_DIRECT_COUNTIES)} 个省直辖县级行政区划")

### 4.2 区域解析函数

In [ ]:
def parse_region_name(region_name, admin_level='District'):
    """
    解析区域名称，提取省、市、区县
    
    Args:
        region_name: 区域名称（如"广东省-广州市-天河区"）
        admin_level: 行政级别
    
    Returns:
        dict: {省, 市, 区县}
    """
    parts = region_name.split('-')
    
    # 省级
    province = parts[0] if len(parts) > 0 else ''
    
    # 处理省直辖县级行政区划
    if is_province_direct_county(region_name):
        return {
            '省': province,
            '市': '省直辖县级行政区划',
            '区县': parts[1] if len(parts) > 1 else ''
        }
    
    # 处理省级
    if admin_level == 'Provincial':
        return {'省': province, '市': '', '区县': ''}
    
    # 处理省本级
    if admin_level == 'Development Of Provincial':
        return {'省': province, '市': '省本级', '区县': ''}
    
    # 处理直辖市
    if is_direct_municipality(region_name):
        if admin_level in ['District', 'Development Of District']:
            return {
                '省': province,
                '市': province,
                '区县': parts[-1] if len(parts) > 1 else ''
            }
        elif admin_level in ['City', 'Development Of City']:
            return {
                '省': province,
                '市': province,
                '区县': parts[1] if len(parts) > 1 else ''
            }
    
    # 处理国家新区（非直辖市）
    if admin_level == 'New District Of National':
        return {
            '省': province,
            '市': parts[1] if len(parts) > 1 else '',
            '区县': '市本级'
        }
    
    # 处理普通地级市（区县级）
    if admin_level == 'District':
        return {
            '省': province,
            '市': parts[1] if len(parts) > 1 else '',
            '区县': parts[-1] if len(parts) > 2 else ''
        }
    
    # 处理市级
    if admin_level in ['City', 'Development Of City']:
        return {
            '省': province,
            '市': parts[1] if len(parts) > 1 else '',
            '区县': '市本级' if admin_level == 'Development Of City' else ''
        }
    
    # 处理区县级开发区
    if admin_level == 'Development Of District':
        return {
            '省': province,
            '市': parts[1] if len(parts) > 1 else '',
            '区县': parts[-1] if len(parts) > 2 else ''
        }
    
    # 默认返回
    return {
        '省': province,
        '市': parts[1] if len(parts) > 1 else '',
        '区县': parts[-1] if len(parts) > 2 else ''
    }

print("区域解析函数定义完成")

### 4.3 测试区域解析

In [ ]:
# 测试用例
test_cases = [
    ("广东省-广州市-天河区", "District"),
    ("北京市-东城区", "District"),
    ("新疆维吾尔自治区-五家渠市", "District"),
    ("湖北省-仙桃市", "District"),
    ("广东省本级", "Development Of Provincial"),
    ("四川省-成都市-天府新区", "New District Of National"),
]

print("区域解析测试结果:")
print("-" * 60)
for region_name, admin_level in test_cases:
    result = parse_region_name(region_name, admin_level)
    print(f"{region_name} ({admin_level})")
    print(f"  -> 省: {result['省']}, 市: {result['市']}, 区县: {result['区县']}")
    print()

---

## 5. 数据可视化与分析

### 5.1 数据预览

In [ ]:
# 读取已采集的数据进行预览
with sql_engine.begin() as connection:
    df_preview = pd.read_sql("SELECT * FROM yq.yyqy LIMIT 10", connection)

print(f"数据预览 (前10行，共 {len(df_preview)} 列):")
df_preview.head()

### 5.2 数据统计

In [ ]:
# 统计数据概况
with sql_engine.begin() as connection:
    # 总记录数
    total_count = pd.read_sql("SELECT COUNT(*) as cnt FROM yq.yyqy", connection).iloc[0]['cnt']
    
    # 行政级别分布
    level_dist = pd.read_sql("""
        SELECT `Administrative Level` as level, COUNT(*) as count 
        FROM yq.yyqy 
        GROUP BY `Administrative Level`
    """, connection)

print(f"总记录数: {total_count}")
print("\n行政级别分布:")
print(level_dist)

---

## 6. 改进建议

### 6.1 添加错误重试机制

In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_retry_session(retries=3, backoff_factor=1):
    """
    创建带重试机制的requests会话
    
    Args:
        retries: 最大重试次数
        backoff_factor: 退避因子
    
    Returns:
        requests.Session: 配置好的会话对象
    """
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session

print("重试机制函数定义完成")

### 6.2 数据验证函数

In [ ]:
def validate_region_data(df):
    """
    验证区域数据完整性
    
    Args:
        df: 数据DataFrame
    
    Returns:
        tuple: (是否有效, 警告消息列表)
    """
    warnings = []
    is_valid = True
    
    # 检查必填字段
    required_fields = ['Administrative Division Name', 'Report Date']
    for field in required_fields:
        if field not in df.columns:
            warnings.append(f"缺少必填字段: {field}")
            is_valid = False
    
    # 检查GDP数值范围
    if '地区生产总值' in df.columns:
        negative_gdp = df[df['地区生产总值'] < 0]
        if len(negative_gdp) > 0:
            warnings.append(f"发现 {len(negative_gdp)} 条GDP为负值的记录")
    
    # 检查重复记录
    if 'Administrative Division Name' in df.columns and 'Report Date' in df.columns:
        duplicates = df.duplicated(subset=['Administrative Division Name', 'Report Date'])
        if duplicates.sum() > 0:
            warnings.append(f"发现 {duplicates.sum()} 条重复记录")
    
    return is_valid, warnings

print("数据验证函数定义完成")

---

## 7. 总结与输出

### 7.1 执行结果汇总

In [ ]:
# 生成执行报告
report = {
    "taskId": "T53",
    "taskName": "yy区域数据整理",
    "executionTime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataSource": "企业预警通(Ratingdog)",
    "targetDatabase": f"{DB_HOST}:{DB_PORT}/{DB_NAME}",
    "targetTable": "yq.yyqy",
    "indicatorCount": len(ALL_INDICATORS),
    "status": "completed"
}

print("=" * 60)
print("执行报告")
print("=" * 60)
for key, value in report.items():
    print(f"{key}: {value}")

### 7.2 后续优化方向

1. **错误处理**：添加更完善的重试机制和日志记录
2. **数据验证**：实现更严格的数据完整性检查
3. **性能优化**：批量插入、并发请求提高效率
4. **增量更新**：只获取新增数据，减少传输量
5. **监控告警**：添加任务执行状态监控

In [ ]:
# 保存执行报告
import json

report_path = "output/execution_report.json"
os.makedirs("output", exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"执行报告已保存至: {report_path}")